# PICKO Research · NB3 — **Separation**: can it tell look-alike tools apart?

Train one **40-tool model**, then probe curated groups of near-identical tools (same action across
sources, or same source across actions). Each group is small enough to offer in full at inference, so we
measure pure disambiguation + which tool it confuses for which.

*Run & forget:* the 40-tool model trains once to Drive and is reused; a restart **skips finished groups**
(`picko_out/separation_results.json`).

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab first: Runtime → Change runtime type → GPU (L4 recommended; T4/A100 also fine).**
This cell clones the repo, pins the exact JAX/Flax, mounts Drive, and points **both** the data (in) and
the checkpoints+results (out) at your **`MyDrive/picko/`** folder — so a runtime restart loses nothing.

**Prerequisite (one-time):** `picko_balanced.jsonl` must be in `MyDrive/picko/`. **Running locally?** This
cell is a no-op — skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/picko"):
        !git clone -b hadar-work https://github.com/HadarBit/picko.git /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    import shutil
    DRIVE = "/content/drive/MyDrive/picko"                      # <- everything lives here
    os.environ["PICKO_OUT_DIR"] = f"{DRIVE}/picko_out"          # checkpoints + results (durable)
    os.environ["PICKO_LOG"]     = f"{DRIVE}/picko_out/run.log"  # durable log across restarts
    os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    dst = "/content/picko/data/picko_balanced.jsonl"
    if not os.path.exists(dst):
        cands = [f"{DRIVE}/picko_balanced.jsonl", "/content/drive/MyDrive/picko_balanced.jsonl"]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            have = os.listdir(DRIVE) if os.path.isdir(DRIVE) else "(MyDrive/picko not found)"
            raise FileNotFoundError(
                "picko_balanced.jsonl not found. Upload it to MyDrive/picko/. "
                f"Currently in {DRIVE}: {have}")
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
        print("copied data from", src)
    import jax
    print("GPU:");
    !nvidia-smi -L
    print("jax devices:", jax.devices())
    _plat = jax.devices()[0].platform
    assert _plat == "gpu", (
        f"JAX is running on '{_plat}', NOT the GPU — every finetune/eval will be ~30x slower "
        "(hours instead of minutes). FIX: Runtime > Change runtime type > GPU (L4), then "
        "Runtime > Restart session, and re-run this cell. If a GPU IS selected but this still "
        "fails, the CUDA plugin didn't load — re-run the %pip line above, then restart.")
    print("bootstrap OK · GPU active · data =", dst, "· OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally (CPU).")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()
env_report(OUT_DIR)   # jax devices + is OUT_DIR durable (Drive)?

### The 40 focus tools\nOne row per tool, with its family, category and **parameter count / bucket**.

In [ ]:
display(tools_dataframe(cat, FOCUS))

### All examples for these 40 tools\nOne row per training example (query → gold tool), tagged with the gold tool's **param bucket**.

In [ ]:
ex_df = examples_dataframe(cat, raw, FOCUS)
print("examples:", ex_df.shape[0], "| per param bucket:", ex_df["param_bucket"].value_counts().to_dict())
display(ex_df.head(10))

## 2 · The ambiguous groups

In [ ]:
grp_rows = []
for g, tools in SIMILAR_GROUPS.items():
    fams = sorted({family_of(t) for t in tools})
    buckets = sorted({param_bucket(cat.params_of(t)[1]) for t in tools})
    grp_rows.append({"group": g, "n_tools": len(tools), "families": ",".join(fams),
                     "param_buckets": ",".join(buckets), "tools": ", ".join(tools)})
groups_df = pd.DataFrame(grp_rows)
display(groups_df)

## 3 · Train (or reuse) the 40-tool model\nResumable: reuses `picko_focus40_best.pkl` from Drive if present.

In [ ]:
CAP_PER_TOOL, EPOCHS, EVAL_SUBSAMPLE, BATCH_SIZE = 40, 1, 40, 8   # BATCH_SIZE: raise to 16 if headroom, lower to 4 on OOM
RUN_TRAIN, FORCE_RETRAIN = True, False
FOCUS40 = finetune_and_eval(cat, raw, tok, FOCUS, "focus40", OUT_DIR,
                            cap=CAP_PER_TOOL, epochs=EPOCHS, compact=False, token_aware=True,
                            eval_subsample=EVAL_SUBSAMPLE, run_train=RUN_TRAIN,
                            force_retrain=FORCE_RETRAIN, batch_size=BATCH_SIZE)
m40, p40, tk40 = FOCUS40["bundle"]
log(f"focus40 overall selection_acc={FOCUS40['metrics']['selection_acc']:.3f}")

## 4 · Per-group disambiguation\n*Resumable:* finished groups are skipped; results persist per group to `OUT_DIR/separation_results.json`.

In [ ]:
RES = os.path.join(OUT_DIR, "separation_results.json")
prev = json.load(open(RES)) if (os.path.exists(RES) and not FORCE_RETRAIN) else {"per_group": [], "confusion": {}}
sep_by = {r["group"]: r for r in prev.get("per_group", [])}
group_conf = prev.get("confusion", {})
if sep_by: log(f"loaded {len(sep_by)} finished group(s) from {RES}")

t_all = time.time()
for gname, gtools in SIMILAR_GROUPS.items():
    if gname in sep_by and gname in group_conf and not FORCE_RETRAIN:
        log(f"{gname}: skip (already done) — selection={sep_by[gname]['selection_acc']:.3f}"); continue
    try:
        log(f"=== group {gname} ({len(gtools)} tools) ===")
        gset = cat.restrict_dataset(raw, gtools, offer_all_max=len(gtools), cap_per_tool=CAP_PER_TOOL, seed=0)
        _, _, gtest = per_tool_split(gset)
        if EVAL_SUBSAMPLE: gtest = gtest[:EVAL_SUBSAMPLE]
        gpreds = predict(m40, p40, tk40, gtest)
        gm = evaluate(gtest, gpreds, family_of=family_of)
        sep_by[gname] = {"group": gname, "n_tools": len(gtools), "n": gm["n"],
                         "selection_acc": gm["selection_acc"], "name_f1": gm["name_f1"]}
        group_conf[gname] = confusion(gtest, gpreds)
        json.dump({"per_group": list(sep_by.values()), "confusion": group_conf}, open(RES, "w"), indent=2)
        log(f"=== done {gname}: selection={gm['selection_acc']:.3f} ===")
    except Exception as e:
        log(f"{gname}: FAILED ({type(e).__name__}: {e}) — skipping; re-run to resume")

log(f"ALL GROUPS DONE in {time.time()-t_all:.0f}s · results={RES}")
if not sep_by:
    raise RuntimeError("No group succeeded — see the FAILED lines above (fix the error, then re-run).")
separation = pd.DataFrame(list(sep_by.values())).sort_values("selection_acc")
display(separation)

plt.figure(figsize=(8,4)); plt.barh(separation["group"], separation["selection_acc"], color="#4C72B0")
plt.xlim(0,1); plt.xlabel("tool-selection accuracy"); plt.title("Separation: hardest look-alike groups (lower = more confused)")
plt.tight_layout(); save_fig("separation_groups"); plt.show()

## 5 · Confusion heatmaps (who gets mistaken for whom)

In [ ]:
for gname, conf in group_conf.items():
    labels = sorted(set(conf) | {p for row in conf.values() for p in row})
    M = pd.DataFrame(0, index=sorted(conf), columns=labels)
    for r, row in conf.items():
        for p, n in row.items(): M.loc[r, p] = n
    plt.figure(figsize=(0.9*len(labels)+2, 0.5*len(M)+1.5))
    if sns: sns.heatmap(M, annot=True, fmt="d", cmap="Blues", cbar=False)
    else:
        plt.imshow(M.values, cmap="Blues"); plt.xticks(range(len(labels)), labels, rotation=90); plt.yticks(range(len(M)), M.index)
    plt.title(f"Separation · {gname}"); plt.xlabel("predicted"); plt.ylabel("reference")
    plt.tight_layout(); save_fig(f"separation_confusion_{gname}"); plt.show()

## 6 · Read-out

Residual selection errors concentrate inside these look-alike groups. The lowest-accuracy group is the
frontier for a tool-picker; the heatmaps show whether confusions are symmetric (two tools mutually
confused) or a sink (everything collapses to one generic tool).